In [1]:
# Loading + chunking
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector store
from langchain_chroma import Chroma

# LLM (local, via Ollama)
from langchain_ollama import OllamaLLM


from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate

from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever


/tmp/ipykernel_164070/1794618111.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


In [2]:
llm = OllamaLLM(model="phi4-mini") 

In [3]:
path = "/home/therealgone/Projects/Brain-Rag/memory.txt"
loader = TextLoader(file_path=path)

docs = loader.load()
for doc in docs:
    print(doc.model_dump())

{'id': None, 'metadata': {'source': '/home/therealgone/Projects/Brain-Rag/memory.txt'}, 'page_content': "My name is Alex and I'm a backend developer working mostly with Python and Go. I've been coding professionally for about 4 years now, starting out at a small fintech startup before moving to a larger e-commerce company last year.\n\nI want to watch the movie Interstellar this weekend. I've heard really good things about the visuals and the sound design. A few of my friends said the ending confused them, so I should probably watch it when I can focus without distractions.\n\nFor my workout routine, I go to the gym three times a week, usually Monday, Wednesday, and Friday. I focus on compound lifts, squats, deadlifts, and bench press. My current goal is to hit a 100kg bench press by the end of the year.\n\nI'm currently learning how to cook Italian food properly. Last week I made a carbonara from scratch and it turned out too watery, I think I added the egg mixture while the pan was t

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name="memories",
    embedding_function=embeddings,
    persist_directory="/home/therealgone/Projects/Brain-Rag/chroma_db",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=100,
    chunk_overlap=0,
)

chunks = splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(chunk.page_content)
    print()
    save_memory(chunk, path, i)

Created a chunk of size 229, which is longer than the specified 100
Created a chunk of size 242, which is longer than the specified 100
Created a chunk of size 231, which is longer than the specified 100
Created a chunk of size 280, which is longer than the specified 100
Created a chunk of size 243, which is longer than the specified 100
Created a chunk of size 259, which is longer than the specified 100


--- Chunk 0 ---
My name is Alex and I'm a backend developer working mostly with Python and Go. I've been coding professionally for about 4 years now, starting out at a small fintech startup before moving to a larger e-commerce company last year.

Stored chunk 0 in Chroma with id='memory.txt_0'
--- Chunk 1 ---
I want to watch the movie Interstellar this weekend. I've heard really good things about the visuals and the sound design. A few of my friends said the ending confused them, so I should probably watch it when I can focus without distractions.

Stored chunk 1 in Chroma with id='memory.txt_1'
--- Chunk 2 ---
For my workout routine, I go to the gym three times a week, usually Monday, Wednesday, and Friday. I focus on compound lifts, squats, deadlifts, and bench press. My current goal is to hit a 100kg bench press by the end of the year.

Stored chunk 2 in Chroma with id='memory.txt_2'
--- Chunk 3 ---
I'm currently learning how to cook Italian food properly. Last week I made a carbona

In [ ]:
# import json , os 

# Categories_path= "/home/therealgone/Projects/Brain-Rag/categories.json"

# try:
#     with open(Categories_path, "r") as f:
#         categories = json.load(f)
#     if not isinstance(categories, list):
#         raise ValueError
# except (FileNotFoundError, json.JSONDecodeError, ValueError):
#     categories = []
#     with open(Categories_path, "w") as f:
#         json.dump(categories, f)

# print(categories)

In [ ]:
# from langchain_core.prompts import PromptTemplate

# categorize_prompt = PromptTemplate.from_template("""
# You are categorizing a personal memory note into a broad topic category.

# Existing categories: {categories}

# Memory text:
# \"\"\"
# {chunk_text}
# \"\"\"

# Instructions:
# - Categories should be broad and general, not specific or narrow. Think top-level life areas, not sub-topics.
#   Good examples: Health, Fitness, Work, Learning, Projects, Travel, Movies, Books, Food, Relationships, Finance.
#   Bad examples (too specific): "Italian Cooking Mistakes", "Interstellar Review", "Bench Press Progress".
# - Always check the existing categories first. If the memory reasonably fits one of them, even loosely, use that exact existing category name rather than creating a similar new one.
# - Only invent a new category if the memory genuinely doesn't fit any existing one. New categories should be 1-2 words, Title Case, and just as broad as the examples above.
# - Respond with ONLY the category name. No explanation, no punctuation, no extra text.
# """)

In [ ]:
# def save_category_if_new(category, categories):
#     if category not in categories:
#         categories.append(category)
#         with open(Categories_path, "w") as f:
#             json.dump(categories, f, indent=2)
#         print(f"Added new category: {category}")
#     else:
#         print(f"Matched existing category: {category}")
#     return categories

# categories = save_category_if_new(category, categories)


In [ ]:
from datetime import datetime
import os

def save_memory(chunk, path, i):
    chunk_id = f"{os.path.basename(path)}_{i}"
    chunk.metadata["id"] = chunk_id 
    chunk.metadata["Date-Time"] = datetime.now().isoformat()
    chunk.metadata["last_edited"] = ""
    chunk.metadata["edit_trail"] = ""
    

   
    vectorstore.add_documents([chunk], ids=[chunk_id])
    print(f"Stored chunk {i} in Chroma with id={chunk_id!r}")

In [13]:
from datetime import datetime
import os
import uuid

def save_memory_chat(chunk):
    chunk.metadata["Date-Time"] = datetime.now().isoformat()
    chunk.metadata["last_edited"] = ""
    chunk.metadata["edit_trail"] = ""

    chunk_id = str(uuid.uuid4())
    vectorstore.add_documents([chunk], ids=[chunk_id])
  

In [ ]:
# import datetime
# llm = OllamaLLM(model="phi4-mini")   # check `ollama list` for the exact tag you pulled

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# vectorstore = Chroma(
#     collection_name="memories",
#     embedding_function=embeddings,
#     persist_directory="/home/therealgone/Projects/Brain-Rag/chroma_db",
# )

# for i, chunk in enumerate(chunks):

#     prompt_text = categorize_prompt.format(
#         categories=categories if categories else "None yet",
#         chunk_text=chunk.page_content,

#     )

#     response = llm.invoke(prompt_text)
#     category = response.strip()
#     print(repr(category))
#     categories = save_category_if_new(category, categories)

#     chunk.metadata["category"] = category
#     chunk.metadata["Date-Time"]= datetime.now().isoformate()
#     chunk.metadata["last_edited"] = None 
#     chunk.metadata["edit_trail"] = ""  
#     chunk_id = f"{os.path.basename(path)}_{i}"

#     vectorstore.add_documents([chunk], ids=[chunk_id])
#     print(f"Stored chunk {i} in Chroma with id={chunk_id!r}, category={category!r}")

In [ ]:
rewrite_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Look at the user's message and decide what it is, then rewrite it accordingly.

First, decide:
- If the message tells you something changed, was updated, is no longer true, or provides new information about something that already exists, treat it as an EDIT.
- Otherwise, if the message is asking for information back, treat it as a QUESTION.

Then rewrite it using the matching rules below.

If EDIT:
Extract a short, specific search phrase naming the exact topic/subject being referred to, so it can be matched against existing stored memories.
- Do NOT include the new information itself, only the subject/topic being referred to.
- Do NOT guess the topic if it isn't clearly named.
- Keep it short, just the subject, not a full sentence.

If QUESTION:
Rewrite the user's question into a clear, grammatically correct, and specific question for retrieval.
- Preserve the exact meaning and intent of the original question.
- Do NOT add facts, topics, entities, context, assumptions, or interpretations that are not explicitly present in the question.
- Do NOT guess what the user means.
- Do NOT answer the question.
- Do NOT expand vague questions with invented context.
- If the original question is already clear, return it with only minor grammatical improvements.
- If the question is vague, improve its wording while keeping the same level of ambiguity.
- Preserve important words, names, entities, and terminology from the original question.
- Only use information contained in the original message.
- The rewritten version may be identical to the original if no meaningful clarification is possible.

Respond in EXACTLY this format, one line only:
EDIT: <search phrase>
or
QUESTION: <rewritten question>

No explanation, no extra text.

Message: {question}

Response:"""
) 

In [ ]:
# def keyword_search(keyword,vectorstore,k=4):
#     all_data = vectorstore.get()  
    
#     documents = all_data["documents"]
#     metadatas = all_data["metadatas"]
    
#     matches = []
#     for doc, meta in zip(documents, metadatas):
#         if keyword.lower() in doc.lower():   
#             matches.append(doc)
    
#     return matches   



               
               


In [9]:
answer_prompt = PromptTemplate.from_template(
    """You are answering a question using the user's personal memory notes below.

Memory excerpts:
{context}

Question: {question}

Instructions:
- Answer using only the information in the memory excerpts above.
- If the excerpts don't contain enough information to answer, say "I don't have a memory about that."
- Be concise and direct. """)

In [79]:
select_id = PromptTemplate.from_template(
    """You are selecting which memory best answers the user's question, then responding with its ID.

Memory candidates:
{context}

Question: {question}

Instructions:
- Choose the ONE candidate that best answers the question.
- If none of the candidates contain relevant information, respond with exactly: NONE
- Respond with ONLY the id value, nothing else — no quotes, no labels, no explanation, no extra text.
- For example, if the candidate is id='value', respond with exactly: value
"""
)

In [ ]:
import json
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

def retrieve(question, k=3):
    # Step 1: rewrite the vague question
    rewrite_text = rewrite_prompt.format(question=question)

    rewritten_question = llm.invoke(rewrite_text).strip()

    print("new q", rewritten_question)
    # Step 2: get keyword + category together, from the rewritten question
    combined_text = select_categories.format(
        # categories=categories if categories else "None yet",
        question=rewritten_question,
    )
    raw_response = llm.invoke(combined_text).strip()

    try:
        parsed = json.loads(raw_response)
        predicted_keyword = parsed.get("keyword", "")
        # predicted_category = parsed.get("category", "")
    except json.JSONDecodeError:
        print("Failed to parse JSON, falling back to unfiltered search")
        predicted_keyword = ""
        # predicted_category = ""

    print("BM")
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

    # bm25 retriever, built from the same chunked documents
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 3

    # combine them, weights control how much each contributes
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.6, 0.4],
        c=20  # tune this, e.g. 0.4/0.6 if one is more reliable
    )

    results_b = ensemble_retriever.invoke(rewritten_question)
    print("BM")

    results = []

    # try category-filtered search first
    # if predicted_category in categories:
    #     results = vectorstore.similarity_search_with_score(
    #         rewritten_question, k=k, filter={"category": predicted_category}
    #     )

    # fall back to keyword search if category search gave nothing

    if  predicted_keyword in results:
        print("keyword")
        results = keyword_search(predicted_keyword, vectorstore, k=k)

    # final fallback, plain unfiltered similarity search
    if not results:
        print("Falling back to unfiltered search")
        results = vectorstore.similarity_search_with_score(rewritten_question, k=k)

# combine both lists into one list of Document objects first
    combined_docs = results_b + [doc for doc, score in results]

# now build context string from page_content
    context = "\n\n".join([doc.page_content for doc in combined_docs])

    response = answer_prompt.format(context=context, question=rewritten_question)
    answer = llm.invoke(response)
    print(answer)



In [ ]:
import json
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

def retrieve_tool(question):
    # Step 1: rewrite the vague question
    rewrite_text = rewrite_prompt.format(question=question)

    rewritten_question = llm.invoke(rewrite_text).strip()

    print("new q", rewritten_question)
  
    

    

    
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

    # bm25 retriever, built from the same chunked documents
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 1

    # combine them, weights control how much each contributes
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.6, 0.4],
        c=20  # tune this, e.g. 0.4/0.6 if one is more reliable
    )

    results_b = ensemble_retriever.invoke(rewritten_question)
    results_s = vectorstore.similarity_search_with_score(rewritten_question, k=1)    
        
    print("------------------------------------")
    print("\n BM:\n",results_b)

    print("------------------------------------")
 
    print("\n Similartiy:\n",results_s)
    print("------------------------------------")

    context = results_s + results_b 
    return context

    # response = select_id.format(context=context, question=rewritten_question)
    # answer = llm.invoke(response)
    # print(answer)

    # if answer == None:
    #     return None

    # existing = vectorstore.get(ids=[answer])

    
    # content = existing["documents"][0]
    # metadata = existing["metadatas"][0]

    # last_edited = metadata["last_edited"]
    # date_time = metadata["Date-Time"]
    # edit_trail = metadata["edit_trail"]
    # print("------------------------------------")
    # print(content)
    # print("------------------------------------")
    # print(last_edited)
    # print("------------------------------------")
    # print(date_time)
    # print("------------------------------------")
    # print(edit_trail)
    # print("------------------------------------")

    # return content,last_edited,date_time,edit_trail



    

   

    


    






   



In [85]:
q="when is my dentist"
retrieve_tool(q)


new q When is my scheduled appointment with the dentist?
------------------------------------

 BM:
 [Document(metadata={'source': '/home/therealgone/Projects/Brain-Rag/memory.txt', 'Date-Time': '2026-08-29T13:12:27.531652', 'last_edited': '', 'edit_trail': ''}, page_content='For my workout routine, I go to the gym three times a week, usually Monday, Wednesday, and Friday. I focus on compound lifts, squats, deadlifts, and bench press. My current goal is to hit a 100kg bench press by the end of the year.'), Document(id='memory.txt_6', metadata={'source': '/home/therealgone/Projects/Brain-Rag/memory.txt', 'edit_trail': '', 'Date-Time': '2026-08-29T13:12:27.627461', 'last_edited': ''}, page_content='I have a dentist appointment coming up on the 15th of next month for a routine cleaning. I keep forgetting to floss regularly, I should actually start doing that daily instead of only before dentist visits.')]
------------------------------------

 Similartiy:
 [(Document(id='memory.txt_6', me

('I have a dentist appointment coming up on the 15th of next month for a routine cleaning. I keep forgetting to floss regularly, I should actually start doing that daily instead of only before dentist visits.',
 '',
 '2026-08-29T13:12:27.627461',
 '')

In [14]:
from langchain.tools import tool
from langchain_core.documents import Document

@tool
def save_chat_memory(content: str) -> str:
    """Save a new piece of information about the user to long-term memory.
    Use this when the user shares a fact, preference, goal, or event worth remembering,
    such as something they're learning, a plan, a personal detail, or a completed task.
    Do not use this for casual chat that has no lasting value.

    Args:
        content: The information to remember, written as a clear, standalone sentence.
        
    """
    doc = Document(
        page_content=content
    )
    save_memory_chat(doc)
    return f"Saved: {content}"

In [15]:
result = save_chat_memory.invoke({"content":"I like coke zero" })
print(result)

check = vectorstore.get(where={"id": {"$ne": ""}}, limit=1)
print(check)

Saved: I like coke zero
{'ids': ['memory.txt_0'], 'embeddings': None, 'documents': ["My name is Alex and I'm a backend developer working mostly with Python and Go. I've been coding professionally for about 4 years now, starting out at a small fintech startup before moving to a larger e-commerce company last year."], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'last_edited': '', 'Date-Time': '2026-08-29T13:12:27.462837', 'source': '/home/therealgone/Projects/Brain-Rag/memory.txt', 'edit_trail': ''}]}


In [ ]:
@tool
def Memory_Trail(memory_id: str, new_content: str, change_summary: str) -> str:
    """Edit an existing memory when the user indicates something has changed, been completed, or is no longer accurate.

Use this when the user's new message updates, corrects, completes, or changes the status of
something already stored in memory — for example, they finished something they were planning,
changed their mind about a preference, corrected a fact, or an existing plan now has a new detail.

Do NOT use this to create a new, unrelated memory — use save_chat_memory for that instead.
Do NOT use this if you are unsure which memory the user is referring to — in that case, do nothing.

Args:
    memory_id: The unique id of the existing memory to update. You must have already identified
        this id from a prior retrieval/search of existing memories — never guess an id.
    new_content: The updated, current version of the memory, written as a clear, standalone
        sentence, replacing the old content entirely (not appended to it).
    change_summary: A single short sentence summarizing what changed, written in past tense,
        e.g. "User completed the Italian cooking course". Do not include any date, timestamp,
        or pipe character yourself — the date is added automatically from stored metadata.
    """

    print(memory_id) 
    print(new_content)
    print(change_summary)

    update_id = vectorstore.get(ids=[memory_id])

    
    print(update_id)
    






In [ ]:
result = Memory_Trail.invoke({
    "memory_id": "some-actual-existing-id-from-chroma",
    "new_content": "User uses MacOS.",
    "change_summary": "User switched from Arch Linux to MacOS."
})
print(result)

In [16]:
all_docs = vectorstore.get()

for doc_id, content, metadata in zip(all_docs["ids"], all_docs["documents"], all_docs["metadatas"]):
    print(doc_id, "-", content, "-", metadata)

memory.txt_0 - My name is Alex and I'm a backend developer working mostly with Python and Go. I've been coding professionally for about 4 years now, starting out at a small fintech startup before moving to a larger e-commerce company last year. - {'edit_trail': '', 'last_edited': '', 'source': '/home/therealgone/Projects/Brain-Rag/memory.txt', 'Date-Time': '2026-08-29T13:12:27.462837'}
memory.txt_1 - I want to watch the movie Interstellar this weekend. I've heard really good things about the visuals and the sound design. A few of my friends said the ending confused them, so I should probably watch it when I can focus without distractions. - {'source': '/home/therealgone/Projects/Brain-Rag/memory.txt', 'Date-Time': '2026-08-29T13:12:27.505907', 'last_edited': '', 'edit_trail': ''}
memory.txt_2 - For my workout routine, I go to the gym three times a week, usually Monday, Wednesday, and Friday. I focus on compound lifts, squats, deadlifts, and bench press. My current goal is to hit a 100k

In [86]:
from langchain_core.prompts import PromptTemplate

test_rewrite_prompt = PromptTemplate.from_template(
    """Look at the user's message below and decide what it is.

If it sounds like the user is providing new information, or telling you something changed, updated, or is no longer true, treat it as an EDIT. Respond with:
EDIT: <a short search phrase naming the topic/subject being referred to>

If it sounds like the user is asking a question, or wants information back, treat it as a QUESTION. Respond with:
QUESTION: <a rewritten, more descriptive version of the question, expanded for better search>

Respond with ONLY one line, in exactly that format. No explanation.

User message: {message}"""
)

test_messages = [
    "what am I learning right now",
    "I switched from Coke to Fanta",
    "when is my dentist appointment",
    "actually I moved my dentist appointment to the 20th",
    "I don't drink coke zero anymore",
]

for msg in test_messages:
    prompt_text = test_rewrite_prompt.format(message=msg)
    result = llm.invoke(prompt_text).strip()
    print(f"IN:  {msg}")
    print(f"OUT: {result}")
    print("-" * 40)

IN:  what am I learning right now
OUT: QUESTION: What subjects or courses am I currently studying?
----------------------------------------
IN:  I switched from Coke to Fanta
OUT: EDIT: beverage preference change
----------------------------------------
IN:  when is my dentist appointment
OUT: QUESTION: What is the date of my upcoming dentist appointment?
----------------------------------------
IN:  actually I moved my dentist appointment to the 20th
OUT: EDIT: Dentist appointment change
----------------------------------------
IN:  I don't drink coke zero anymore
OUT: EDIT: Coke Zero Consumption Changes
----------------------------------------
